In [ ]:
import cv2
from pathlib import Path
import random
import matplotlib.pyplot as plt
import pandas as pd
import json

In [ ]:
def load_batches_from_summary(summary_path: Path) -> list[dict]:
    """
    Flatten the JSON summary into a unique list of batch dicts:
    Each entry is expected to have:
      - name
      - date
      - image_count
      - upload_only
      - lts
      - seasons (list)
    """
    with open(summary_path, "r") as f:
        summary = json.load(f)

    batches_by_key: dict[tuple[str, str], dict] = {}

    for state, s_info in summary["states"].items():
        # Seasons
        for season_info in s_info["seasons"].values():
            for b in season_info["batches"]:
                key = (state, b["name"])
                b_with_state = dict(b)
                b_with_state.setdefault("state", state)
                batches_by_key[key] = b_with_state

        # Unassigned
        for b in s_info["unassigned"]["batches"]:
            key = (state, b["name"])
            b_with_state = dict(b)
            b_with_state.setdefault("state", state)
            batches_by_key[key] = b_with_state

    batches = list(batches_by_key.values())
    return batches

summary_path = Path("bbotv31_batches.json")
batches = load_batches_from_summary(summary_path)
df = pd.DataFrame(batches)



## ReProcessing old batches results

In [ ]:
df = pd.read_csv("../reprocessed_images_log.csv")
# creaet a index column
# df.reset_index(inplace=True)
print(f"Unique batches in the results: {df['batch'].nunique()}")
print(f"Batches: {df['batch'].unique().tolist()}")
print(f"Total images processed: {df['image_stem'].nunique()}")
print(f"Total images processed: {df['raw_filename'].nunique()}")
print(f"Total images processed: {df.shape[0]}")

# find duplicates
duplicates = df[df.duplicated(subset=['raw_filename'], keep=False)]
if not duplicates.empty:
    print("Duplicate raw filenames found:")
    print(duplicates[['raw_filename', 'batch', 'image_stem']])
    # print index of duplicates
    print("Indices of duplicate entries:")
    print(duplicates.index.tolist())
    # print row number
    # print("Row numbers of duplicate entries:")
    # print((duplicates.index + 2).tolist())  # +2 for header and 0-indexing
duplicates

# calculate the total amount of time using processed_at
df['processed_at'] = pd.to_datetime(df['processed_at'])
df = df.sort_values(by='processed_at')
total_time = (df['processed_at'].iloc[-1] - df['processed_at'].iloc[0]).total_seconds()
print(f"Total processing time: {total_time} seconds")   
# convert to hours
print(f"Total processing time: {total_time/3600} hours")
# average time per image
avg_time_per_image = total_time / df.shape[0]
print(f"Average time per image: {avg_time_per_image} seconds")

In [ ]:
# get a single raw_path
Path("/project/dash_agir/matthew.kutugata/semifield-developed-images/MD_2025-04-14/images/MD_1744658384.jpg")

# Get a single image from each unique batch. Use groupby and sample one row from each group
sampled_df = df.groupby('batch').apply(lambda x: x.sample(5)).reset_index(drop=True)

for index, row in sampled_df.iterrows():
    jpg_path = Path(row['jpg_path'])
    
    print(f"JPG path: {jpg_path}")
    jpg_image = cv2.imread(str(jpg_path))
    # create a cetner crop 1000x1000
    center_y = jpg_image.shape[0] // 2
    center_x = jpg_image.shape[1] // 2
    half_size = 500
    cropped_image = jpg_image[center_y - half_size:center_y + half_size, center_x - half_size:center_x + half_size]

    # save to output folder
    # resize to 1/4
    jpg_image_resized = cv2.resize(jpg_image, (jpg_image.shape[1] // 4, jpg_image.shape[0] // 4))

    # paste the cropped image to the center of the resized image
    start_y = (jpg_image_resized.shape[0] - cropped_image.shape[0]) // 2
    start_x = (jpg_image_resized.shape[1] - cropped_image.shape[1]) // 2
    jpg_image_resized[start_y:start_y + cropped_image.shape[0], start_x:start_x + cropped_image.shape[1]] = cropped_image
    # save to output folder
    output_path = Path("sampled_jpgs") / f"{jpg_path.stem}_resized.jpg"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), jpg_image_resized)

## Image processing visual results

In [ ]:
img_dir = sorted(Path("/mnt/research-projects/s/screberg/longterm_images2/semifield-developed-images/").glob("*"))
img_dir = [p for p in img_dir if "2025-10" in p.name]
img_dir


In [ ]:
img_dir = Path("/mnt/research-projects/s/screberg/longterm_images2/semifield-developed-images/MD_2025-10-03/images")
imgs = sorted(img_dir.glob("*.jpg"))
print(f"Total images: {len(imgs)}")


In [ ]:
plt.close('all')

sample_n = 7
paths = sorted(Path("/project/dash_agir/matthew.kutugata/semifield-developed-images/MD_2025-09-09/images").glob("*.jpg"))
for img_path in random.sample(paths, min(sample_n, len(paths))):
    print(img_path)
    img = cv2.imread(str(img_path))
    # reszie by 0.25 because images are too large
    img = cv2.resize(img, (0,0), fx=0.25, fy=0.25)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis('off')
    plt.show()
# clear all plots
plt.close('all')


In [ ]:
img_path = Path("/project/dash_agir/matthew.kutugata/semifield-developed-images/MD_2025-04-14/images/MD_1744658384.jpg")
img = cv2.imread(str(img_path))
# reszie by 0.25 because images are too large
img = cv2.resize(img, (0,0), fx=0.25, fy=0.25)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img)
plt.title(img_path.name)
plt.axis('off')
plt.show()

In [ ]:

""